In [1]:
from qiskit.circuit import Parameter, QuantumCircuit, QuantumRegister, ClassicalRegister

from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector, Operator
from scipy.optimize import minimize 
from qiskit.circuit.library import QFT
from qiskit import transpile
from qiskit.circuit.library import UnitaryGate

import random
import matplotlib.pyplot as plt
import scipy.linalg as scl
import numpy as np
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeKyiv
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
backend=AerSimulator()
# backend=FakeKyiv()
# sampler = Sampler(backend = backend)
pm = generate_preset_pass_manager(backend = backend,
                                  optimization_level = 2)

In [2]:
nb_qubits = 4
depth = 2
qubits = list(range(nb_qubits))
N = len(qubits)
nb_params = int(9*N*depth)

shots = 100000

parameters = np.array([random.random() for _ in range(0, nb_params)])

def ansatz(parameters):
    qc = QuantumCircuit(N)
    for d in range(depth):
        param1=parameters[d*9*N:(d+1)*(9*N)]
        for q in range(N):
            qc.ry(param1[q],qubits[q])
            qc.ry(param1[q+N],qubits[q])
            qc.ry(param1[q+2*N],qubits[q])
        qc.barrier()
        for q in range(N):
            qc.cx(qubits[q], qubits[(q+1)% N])
            qc.ry(param1[q+3*N],qubits[q])
            qc.ry(param1[q+4*N],qubits[(q+1)% N])
            qc.cx(qubits[(q+1)% N], qubits[q])
            qc.ry(param1[q+5*N],qubits[(q+1)% N])
            qc.cx(qubits[q], qubits[(q+1)% N])
        qc.barrier()
        if d==depth-1:
            for q in range(N):
                qc.ry(param1[q+6*N],qubits[q])
                qc.ry(param1[q+7*N],qubits[q])
                qc.ry(param1[q+8*N],qubits[q])
        qc.barrier()
    
    return qc  


Nd = 2**nb_qubits
m = np.zeros((Nd,Nd))
for j in range(Nd):
    if j == Nd-1:
        break
    else:
       m[j,j+1] = -1 

for j in range(Nd):
    if j == Nd-1:
        break
    else:
       m[j+1,j] = -1 
for j in range(Nd):
   m[j,j] = 2 
m[0] = np.array([1]+ [0]*(Nd-1))
m[1,0] = 0
if nb_qubits == 2:
    b = np.array([0,np.sqrt(2)/2,0.5,0.5])
    # b = np.array([1,1,1,1])/2
elif nb_qubits == 3:
    b = np.array([0,0.25,0.25,0.25,0.25,0.5,0.5,0.5])
else:
    b = np.array([1]*2**nb_qubits)
    b = b/np.linalg.norm(b)

In [3]:
    
def U_b(nb_qubits):
    circ = QuantumCircuit(nb_qubits)
    circ.prepare_state(b)
    return circ
U = U_b(nb_qubits)
b = np.array(Statevector(U_b(nb_qubits)))
b

array([0.25+0.j, 0.25+0.j, 0.25+0.j, 0.25+0.j, 0.25+0.j, 0.25+0.j,
       0.25+0.j, 0.25+0.j, 0.25+0.j, 0.25+0.j, 0.25+0.j, 0.25+0.j,
       0.25+0.j, 0.25+0.j, 0.25+0.j, 0.25+0.j])

In [4]:
def Hamiltonian(m,b):
    op = np.zeros((2**nb_qubits,2**nb_qubits), dtype = complex)
    for k in range(nb_qubits):
        qc = QuantumCircuit(nb_qubits)
        qc.z(k)
        op += np.array(Operator(qc))
    u = np.identity(2**nb_qubits) - op/nb_qubits
    Ub = np.array(Operator(U_b(nb_qubits)))
    return 0.5*np.conj(m.T)@Ub@u@np.conj(Ub.T)@m
A = Hamiltonian(m,b)
np.linalg.eigvals(A)


array([9.95669810e+00+0.j, 9.28822592e+00+0.j, 7.10726222e+00+0.j,
       4.57560921e+00+0.j, 5.80071269e+00+0.j, 5.43799224e+00+0.j,
       2.92422042e+00+0.j, 1.64735275e+00+0.j, 1.06298356e+00+0.j,
       9.56720841e-01+0.j, 4.55576346e-01+0.j, 3.78140696e-01+0.j,
       1.02591714e-01+0.j, 4.92365599e-02+0.j, 6.67673988e-03+0.j,
       8.71777968e-16+0.j])

In [5]:

def Optimizer(fun, x0, args=(), maxfev=None, reset_interval=None, eps=None, callback=None, **_):
    
    x0 = np.asarray(x0)
    recycle_z0 = None
    niter = 0
    funcalls = 0

    while True:

        idx = niter % x0.size

        if reset_interval > 0:
            if niter % reset_interval == 0:
                recycle_z0 = None

        if recycle_z0 is None:
            z0 = fun(np.copy(x0), *args)
            funcalls += 1
        else:
            z0 = recycle_z0

        p = np.copy(x0)
        p[idx] = x0[idx] + np.pi / 2
        z1 = fun(p, *args)
        funcalls += 1

        p = np.copy(x0)
        p[idx] = x0[idx] - np.pi / 2
        z3 = fun(p, *args)
        funcalls += 1

        z2 = z1 + z3 - z0
        c = (z1 + z3) / 2
        a = np.sqrt((z0 - z2) ** 2 + (z1 - z3) ** 2) / 2
        b = np.arctan((z1 - z3) / ((z0 - z2) + 1e-32 * (z0 == z2))) + x0[idx]
        b += 0.5 * np.pi + 0.5 * np.pi * np.sign((z0 - z2) + eps * (z0 == z2))
        x0[idx] = b
        recycle_z0 = c - a
        if callback is not None:
            callback(np.copy(x0))
        if funcalls >= maxfev:
            break
        niter += 1
    # return OptimizeResult(fun=problabel0(np.copy(x0)), x=x0, nit=niter, 
    #                       nfev=funcalls, success=(niter > 1))

In [6]:
1/2**8

0.00390625

In [7]:
Nx = [2, 4, 6, 8, 10]#number of qubits in the measurement register
RMSE = []
for nx in Nx:
    parameters = np.array([ 4.59991317e+00,  3.59951559e-02, -4.46172707e-01,  5.35559258e+00,
        1.31371984e+00,  1.06587748e+00,  1.05286989e+00, -4.14761103e-01,
        8.81004839e-01,  5.33656135e-01,  4.97468659e-01,  1.79192894e-01,
        2.04364811e-01,  8.54905378e-01, -4.81656229e-02, -4.23508303e-02,
        8.84862717e-01,  5.54591644e-01,  3.66402167e-01,  1.97149856e-01,
        8.21721654e-01,  1.32285264e+00,  1.84813142e-01,  3.41612537e-01,
        2.64973415e+01,  1.84015780e+01,  8.80207317e+00,  1.98889417e-01,
        4.96323360e+00,  2.24407661e-01,  5.81431765e+00, -2.15664697e+00,
        1.53808860e+01,  2.18070036e+00,  4.35407167e+00,  9.24191439e-01,
        5.56062235e-01, -3.06833531e-02,  4.36764635e-01,  8.43671271e-01,
        8.99743320e-01,  4.22013529e-01,  2.91729005e-01,  5.42593252e-01,
        9.05461525e-02,  1.49826017e-01,  8.66106221e-01,  4.56341434e-01,
        6.02326832e-01,  3.73187195e-02, -6.70182890e-01,  4.27600917e-01,
        6.85164753e-01,  3.45873155e-01,  2.50849368e-01,  8.33261727e-01,
        8.37197178e-02,  8.54355890e-01,  5.58187531e-01,  6.06143506e-01,
        7.78658380e-01, -2.23900582e-02,  1.20355257e+00,  2.81681631e-01,
        6.93602378e-01,  7.95256824e-02,  8.12335573e-01,  3.81652752e-01,
        3.67230244e-01,  8.29019351e-01,  9.90061727e-01,  7.17431965e-01])
    CUgate = []
    for k in range(nx):  
        U = scl.expm(2**k*2*np.pi*1j*A) 
        ugate = UnitaryGate(U)
        Cu = ugate.control(annotated=True)
        CUgate.append(Cu)


    def circ(parameters,nx):
        x = QuantumRegister(nx + nb_qubits)
        c = ClassicalRegister(nx)
        circuit = QuantumCircuit(x,c)
        circuit = circuit.compose(ansatz(parameters),x[nx:nx+nb_qubits])
        circuit.h(x[0:nx])
        for k in range(nx):
            circuit.append(CUgate[k], [x[k]] + x[nx:nx+nb_qubits])    
        circuit &= QFT(num_qubits = nx, approximation_degree = 0, do_swaps = True, 
                       inverse = True, insert_barriers = False, name = 'qft')
        circuit.measure(x[0:nx],c)       
        return circuit

    def cost(parameters):
        job = backend.run(pm.run(circ(parameters,nx)),shots = shots).result()
        result = job.get_counts(0)
        if '0'*nx not in result:
            res = 1
        else:
            res = 1 - result['0'*nx]/shots
        return res
    # cost(parameters)

    
    def save(parameters):
        global Cost,Params
        Cost.append(cost(parameters))
        Params.append(parameters)
        # print(cost(parameters))
    Cost = []
    Params = []
    Optimizer(cost, parameters, args = (), maxfev = 4000, 
              reset_interval = 32, eps = 1e-32, callback=save)

    e = []
    F = []
    norm_e = []
    
    x_exact = np.linalg.solve(m,b)
    x_exact = x_exact/np.linalg.norm(x_exact)
    for k in range(len(Params)):
        state = np.array(Statevector(ansatz(Params[k])))
        norm = np.dot(state,x_exact)
        e.append(x_exact - state/norm)
        f = abs(np.dot(state,x_exact))**2
        F.append(f)
    
    for v in e:
        norm_e.append(float(np.linalg.norm(v)))
    Res = norm_e[np.argmax(F)]
    print(Res)
    RMSE.append(Res)

/tmp/ipykernel_1387316/4221363523.py:38: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit &= QFT(num_qubits = nx, approximation_degree = 0, do_swaps = True,


0.010536098124193179
0.006351521953953185
0.0026966024963250785
0.002312328165557386
0.0030127985330907357


In [8]:
print(RMSE)

[0.010536098124193179, 0.006351521953953185, 0.0026966024963250785, 0.002312328165557386, 0.0030127985330907357]
